In [27]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder

silver_path = "../data/silver/"
models_path = "../models/"
os.makedirs(models_path, exist_ok=True)

print("⏳ Reconstruyendo dataset de emergencia directo de Silver...")
# 1. Cargar las tablas de la capa Silver
df_cli = pd.read_parquet(f"{silver_path}clientes.parquet")
df_est = pd.read_parquet(f"{silver_path}cliente_estado_mensual.parquet")

# Unir por id_cliente
df_merged = pd.merge(df_est, df_cli, on='id_cliente', how='inner')

# Convertimos primero id_segmento a string para evitar el AttributeError, o validamos numéricamente
df_merged['id_segmento_str'] = df_merged['id_segmento'].astype(str)
df_merged['y'] = np.where(df_merged['id_segmento_str'].str.contains('1', na=False), 1, 0)
df_merged = df_merged.drop(columns=['id_segmento_str'])

# Codificar variables categóricas de forma segura
columnas_categoricas = ['sexo', 'ind_empleado', 'pais_residencia', 'canal_entrada', 'id_segmento', 'cod_prov']
encoders = {}

for col in columnas_categoricas:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype(str).fillna('MISSING')
        le = LabelEncoder()
        df_merged[col] = le.fit_transform(df_merged[col])
        encoders[col] = le

joblib.dump(encoders, f"{models_path}encoders_nbp.pkl")

# Definir Features de entrenamiento
columnas_excluir = ['id_cliente', 'fecha_corte', 'y']
features = [col for col in df_merged.columns if col not in columnas_excluir]

# Separar los conjuntos de datos de forma segura (80% / 20%)
X_train, X_val, y_train, y_val = train_test_split(
    df_merged[features], 
    df_merged['y'], 
    test_size=0.2, 
    random_state=42,
    stratify=df_merged['y']
)

print("\n ¡EL DATASET DE EMERGENCIA FUE RECONSTRUIDO CON ÉXITO!")
print(f" Las filas reales encontradas para Entrenar (X_train): {X_train.shape}")
print(f" Las filas reales encontradas para Validar (X_val): {X_val.shape}")
print(f" Los casos positivos (y=1) en el entrenamiento: {y_train.sum()}")
print(f"Variables predictoras listas:\n{features}")

⏳ Reconstruyendo dataset de emergencia directo de Silver...

 ¡EL DATASET DE EMERGENCIA FUE RECONSTRUIDO CON ÉXITO!
 Las filas reales encontradas para Entrenar (X_train): (1600, 28)
 Las filas reales encontradas para Validar (X_val): (400, 28)
 Los casos positivos (y=1) en el entrenamiento: 313
Variables predictoras listas:
['origen_datos', 'es_test', 'age', 'antiguedad', 'ind_nuevo', 'indrel', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'ind_actividad_cliente', 'renta', 'cod_prov', 'id_segmento', 'fecha_carga_x', 'archivo_origen_x', 'capa_origen_x', 'ind_empleado', 'pais_residencia', 'sexo', 'fecha_alta', 'canal_entrada', 'indfall', 'aparece_en_train', 'aparece_en_test', 'fecha_carga_y', 'archivo_origen_y', 'capa_origen_y']


In [28]:
# 1. Identificar columnas categóricas (tipo object o category)
columnas_categoricas = ['sexo', 'ind_empleado', 'pais_residencia', 'canal_entrada', 
                        'indfall', 'id_segmento', 'cod_prov', 'categoria_producto_candidato']

# Guardaremos los encoders para poder usarlos en el notebook de predicción después
encoders = {}

for col in columnas_categoricas:
    if col in df_train_full.columns:
        # Rellenar nulos residuales por seguridad antes de codificar
        df_train_full[col] = df_train_full[col].astype(str).fillna('MISSING')
        
        le = LabelEncoder()
        df_train_full[col] = le.fit_transform(df_train_full[col])
        encoders[col] = le

# Guardar los encoders entrenados en la carpeta de modelos
joblib.dump(encoders, f"{models_path}encoders_nbp.pkl")
print(" Las variables categóricas codificadas y encoders guardados.")

 Las variables categóricas codificadas y encoders guardados.


In [29]:
# FILTRADO, LIMPIEZA DE TEXTO Y NULOS

print(" Limpieza de textos y rellenando valores faltantes (NaN)...")

# 1. Aplicar One-Hot Encoding automático a cualquier columna de texto restante
X_train_numeric = pd.get_dummies(X_train, drop_first=True)
X_val_numeric = pd.get_dummies(X_val, drop_first=True)

# 2. Alinear los datasets para asegurarnos de que tengan las mismas columnas
X_train_numeric, X_val_numeric = X_train_numeric.align(X_val_numeric, join='left', axis=1, fill_value=0)

# 3. Filtrar para quedarnos únicamente con columnas de números
X_train_numeric = X_train_numeric.select_dtypes(include=[np.number])
X_val_numeric = X_val_numeric.select_dtypes(include=[np.number])

# 4. Rellenar cualquier valor nulo remanente con 0 para que los modelos no fallen
X_train_numeric = X_train_numeric.fillna(0)
X_val_numeric = X_val_numeric.fillna(0)

print(f" Matrices validadas sin nulos. Columnas finales: {X_train_numeric.shape[1]}")



 Limpieza de textos y rellenando valores faltantes (NaN)...
 Matrices validadas sin nulos. Columnas finales: 16


In [30]:
# ENTRENAMIENTO DE LOS MODELOS

# --- MODELO BÁSICO: Regresión Logística ---
print("\n--- 1. Entrenando Regresión Logística... ---")
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_numeric, y_train)

# Predicciones de probabilidad para la clase 1 (adquisición)
preds_lr = model_lr.predict_proba(X_val_numeric)[:, 1]
auc_roc_lr = roc_auc_score(y_val, preds_lr)
auc_pr_lr = average_precision_score(y_val, preds_lr)

print(f" Regresión Logística lista -> AUC-ROC: {auc_roc_lr:.4f} | Average Precision: {auc_pr_lr:.4f}")

# --- MODELO INTERMEDIO/AVANZADO: Random Forest ---
print("\n--- 2. Entrenando Random Forest Classifier... ---")
model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
model_rf.fit(X_train_numeric, y_train)

# Predicciones de probabilidad
preds_rf = model_rf.predict_proba(X_val_numeric)[:, 1]
auc_roc_rf = roc_auc_score(y_val, preds_rf)
auc_pr_rf = average_precision_score(y_val, preds_rf)

print(f" Random Forest listo -> AUC-ROC: {auc_roc_rf:.4f} | Average Precision: {auc_pr_rf:.4f}")


# --- SELECCIÓN Y ALMACENAMIENTO DEL GANADOR ---
if auc_roc_rf > auc_roc_lr:
    best_model = model_rf
    nombre_ganador = "Random Forest"
else:
    best_model = model_lr
    nombre_ganador = "Regresión Logística"

print(f"\n Este es el modelo ganador por AUC-ROC es: {nombre_ganador}")

# Guardar el archivo binario del modelo en la carpeta models/
joblib.dump(best_model, f"{models_path}modelo_next_best_product.pkl")
print(f" ¡Modelo fue exportado con éxito en: {models_path}modelo_next_best_product.pkl!")


--- 1. Entrenando Regresión Logística... ---
 Regresión Logística lista -> AUC-ROC: 0.9639 | Average Precision: 0.9348

--- 2. Entrenando Random Forest Classifier... ---
 Random Forest listo -> AUC-ROC: 1.0000 | Average Precision: 1.0000

 Este es el modelo ganador por AUC-ROC es: Random Forest
 ¡Modelo fue exportado con éxito en: ../models/modelo_next_best_product.pkl!
